In [18]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

In [19]:
# Resolve dataset location for local runs and Kaggle runs
def resolve_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/cancer-multi-omics-benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Aligned"),
        Path("/kaggle/input/cancer-multi-omics-benchmark/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Aligned"),
        Path("Main_Dataset/Classification_datasets/GS-BRCA/Aligned"),
        Path("Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Aligned"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find GS-BRCA aligned data. Attach the dataset in Kaggle or run from the repo root."
    )

DATA_ROOT = resolve_data_root()

omics_file_1 = pd.read_csv(DATA_ROOT / "BRCA_mRNA_aligned.csv", index_col=0)
omics_file_2 = pd.read_csv(DATA_ROOT / "BRCA_miRNA_aligned.csv", index_col=0)
omics_file_3 = pd.read_csv(DATA_ROOT / "BRCA_CNV_aligned.csv", index_col=0)
omics_file_4 = pd.read_csv(DATA_ROOT / "BRCA_Methy_aligned.csv", index_col=0)
labels = pd.read_csv(DATA_ROOT / "BRCA_label_num.csv").squeeze("columns")

In [20]:
omics_data_1 = omics_file_1.T  # mRNA
omics_data_2 = omics_file_2.T  # miRNA
omics_data_3 = omics_file_3.T  # CNV
omics_data_4 = omics_file_4.T  # Methylation

omics_data_1 = omics_data_1.add_suffix("_mRNA")
omics_data_2 = omics_data_2.add_suffix("_miRNA")
omics_data_3 = omics_data_3.add_suffix("_CNV")
omics_data_4 = omics_data_4.add_suffix("_Methy")

omics_data_combined = omics_data_1.join(omics_data_2, how='inner')
omics_data_combined = omics_data_combined.join(omics_data_3, how='inner')
omics_data_combined = omics_data_combined.join(omics_data_4, how='inner')
omics_data_combined = omics_data_combined.fillna(omics_data_combined.mean())

In [21]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, VarianceThreshold
import matplotlib.pyplot as plt

# ==================== FEATURE SELECTION MODULE ====================

def select_features_kbest(X, y, k=100, score_func='f_classif', verbose=True):
    """
    Select top K features using SelectKBest.
    
    Parameters:
    -----------
    X : DataFrame or ndarray
        Feature matrix
    y : Series or ndarray
        Target labels
    k : int
        Number of top features to select
    score_func : str
        'f_classif' - ANOVA F-value (default)
        'mutual_info' - Mutual information (captures non-linear relationships)
    verbose : bool
        Print feature names and scores
    
    Returns:
    --------
    X_selected : ndarray
        Selected features
    selected_features : list
        Names of selected features
    selector : SelectKBest object
    """
    
    # Choose scoring function
    if score_func == 'mutual_info':
        score_fn = mutual_info_classif
    else:
        score_fn = f_classif
    
    selector = SelectKBest(score_func=score_fn, k=min(k, X.shape[1]))
    X_selected = selector.fit_transform(X, y)
    
    # Get feature names
    if isinstance(X, pd.DataFrame):
        feature_names = X.columns[selector.get_support()]
        selected_features = feature_names.tolist()
    else:
        selected_features = selector.get_support(indices=True).tolist()
    
    if verbose:
        scores = selector.scores_
        feature_score_pairs = sorted(zip(selected_features, scores[selector.get_support()]), 
                                     key=lambda x: x[1], reverse=True)
        print(f"\n📊 Top {len(selected_features)} Features (using {score_func}):")
        print("-" * 60)
        for feat, score in feature_score_pairs[:10]:  # Show top 10
            print(f"  {feat:40s} → Score: {score:.4f}")
        print(f"  ... and {len(selected_features) - 10} more features")
    
    return X_selected, selected_features, selector


def select_features_variance(X, threshold=0.01, verbose=True):
    """
    Remove features with low variance.
    
    Parameters:
    -----------
    X : DataFrame or ndarray
        Feature matrix
    threshold : float
        Variance threshold (default 0.01)
    verbose : bool
        Print number of features removed
    
    Returns:
    --------
    X_selected : ndarray
        Features after variance filtering
    selected_features : list
        Names of remaining features
    """
    
    selector = VarianceThreshold(threshold=threshold)
    X_selected = selector.fit_transform(X)
    
    if isinstance(X, pd.DataFrame):
        selected_features = X.columns[selector.get_support()].tolist()
    else:
        selected_features = selector.get_support(indices=True).tolist()
    
    if verbose:
        removed = X.shape[1] - X_selected.shape[1]
        print(f"\n🔍 Variance Threshold Filter (threshold={threshold}):")
        print(f"  Removed: {removed} low-variance features")
        print(f"  Remaining: {X_selected.shape[1]} features")
    
    return X_selected, selected_features, selector


# ==================== APPLY FEATURE SELECTION ====================

# Choose your feature selection method:
# Option 1: SelectKBest (RECOMMENDED for your dataset)
print("=" * 60)
print("🚀 FEATURE SELECTION")
print("=" * 60)

# Select top 500 features using ANOVA F-value
X_selected, selected_features, selector = select_features_kbest(
    omics_data_combined, 
    labels, 
    k=500,  # Adjust this number as needed
    score_func='f_classif',
    verbose=True
)

# Option 2 (Alternative): Variance threshold + SelectKBest
# X_var_filtered, _, var_selector = select_features_variance(omics_data_combined, threshold=0.01)
# X_selected, selected_features, kbest_selector = select_features_kbest(
#     X_var_filtered, labels, k=500, score_func='mutual_info'
# )

# Create new DataFrame with selected features
omics_data_selected = pd.DataFrame(
    X_selected,
    columns=selected_features,
    index=omics_data_combined.index
)

print(f"\n✅ Feature selection complete!")
print(f"   Original shape: {omics_data_combined.shape}")
print(f"   Selected shape: {omics_data_selected.shape}")
print(f"   Features retained: {(omics_data_selected.shape[1] / omics_data_combined.shape[1] * 100):.1f}%")

🚀 FEATURE SELECTION


ValueError: Input X contains NaN.
SelectKBest does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    omics_data_selected,  # Use selected features instead of combined
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
    shuffle=True,
)

# XGBoost GPU training is faster with contiguous float32 arrays.
X_train = np.ascontiguousarray(X_train.to_numpy(dtype=np.float32))
X_test = np.ascontiguousarray(X_test.to_numpy(dtype=np.float32))
y_train = y_train.to_numpy(dtype=np.int32)
y_test = y_test.to_numpy(dtype=np.int32)

print(f"\n📈 Train-Test Split:")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_test shape: {X_test.shape}")

In [ ]:
param_dist = {
    "n_estimators": [400, 700, 1000, 1400],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0.0, 0.1, 0.3],
    "reg_alpha": [0.0, 0.01, 0.1, 0.5],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
    "max_bin": [256, 384, 512],
}

n_iter_search = 48

In [ ]:
num_class = int(np.unique(y_train).size)

xgb = XGBClassifier(
    n_estimators = 400,
    max_depth = 7,
    learning_rate = 0.01,
    reg_alpha = 0.01,
    random_state=42,
    objective="multi:softprob",
    num_class=num_class,
    tree_method="hist",
    device="cuda",
    eval_metric="mlogloss",
    sampling_method="gradient_based",
    n_jobs=1,
    verbosity=1,
)


In [ ]:
start_time = time.time()
xgb.fit(X_train, y_train)
fit_seconds = time.time() - start_time

KeyboardInterrupt: 

In [ ]:
# ==================== VISUALIZE FEATURE IMPORTANCE ====================

# Get feature importance from trained model
feature_importance = xgb.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

# Plot top 20 important features
plt.figure(figsize=(10, 6))
top_n = 20
top_features = feature_importance_df.head(top_n)
plt.barh(range(len(top_features)), top_features['importance'].values)
plt.yticks(range(len(top_features)), top_features['feature'].values)
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title(f'Top {top_n} Most Important Features (XGBoost)')
plt.tight_layout()
plt.show()

print(f"\n🎯 Top 10 Most Important Features:")
print("-" * 50)
for idx, row in feature_importance_df.head(10).iterrows():
    print(f"  {row['feature']:40s} → {row['importance']:.6f}")

In [ ]:
print(f"Search finished in {fit_seconds / 60:.2f} minutes")

Search finished in 45.15 minutes


In [ ]:
y_pred = xgb.predict(X_test)
print(classification_report(y_true=y_test, y_pred=y_pred, digits=4))

              precision    recall  f1-score   support

           0     0.8800    0.9296    0.9041        71
           1     0.8750    0.8750    0.8750         8
           2     0.8636    0.7037    0.7755        27
           3     0.5000    0.5000    0.5000         6
           4     0.9583    1.0000    0.9787        23

    accuracy                         0.8741       135
   macro avg     0.8154    0.8017    0.8067       135
weighted avg     0.8729    0.8741    0.8714       135

